# Flower Distributed FL — MNIST Orchestrator

This notebook runs the same distributed federation as `Orchestrator.ipynb`,
but trains a **CNN on MNIST** instead of logistic regression on synthetic data.

## Model architecture

```
Input (1×28×28)
  Conv2d(1→32, k=3) → ReLU
  Conv2d(32→64, k=3) → ReLU → MaxPool2d(2) → Dropout(0.5)
  Flatten → Linear(9216→128) → ReLU → Dropout(0.5)
  Linear(128→10)   ← 10 digit classes
```

Each SuperNode trains on a **non-overlapping slice of the MNIST training set**
(30 000 samples per node with 2 nodes). MNIST is downloaded automatically on
first run to `~/.flwr_demo/mnist/`.

## Before running

1. Install PyTorch in your environment if not already present:
   ```bash
   pip install torch torchvision
   ```
2. On the **SuperLink machine**: `bash start_superlink.sh`
3. On each **SuperNode machine**: `bash start_supernode.sh <SUPERLINK_IP>`
4. Set `SUPERLINK_IP` below and run all cells.

---
## 0 - Imports & helpers

In [1]:
import importlib
import workshop_helpers

importlib.reload(workshop_helpers)
from workshop_helpers import (
    bin_dir,
    preflight,
    print_port_status,
    run_stream,
    start_background,
    stop_background,
    update_superlink_address,
)

ctx = preflight("fl_app_mnist")
flwr_bin_prefix = bin_dir(ctx)

Python executable : /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/.venv-workshop/bin/python
Python version    : 3.13.5
Notebook cwd      : /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos
Flower app        : /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/fl_app_mnist
flwr binary       : /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/.venv-workshop/bin/flwr


flwr            : 1.30.0 @ /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/.venv-workshop/lib/python3.13/site-packages/flwr
numpy           : 2.4.6 @ /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/.venv-workshop/lib/python3.13/site-packages/numpy


The helper functions above keep notebook setup out of the way. The remaining cells show the Flower commands in the same shape students would run from a terminal.

---
## 1 — Configuration

Set `SUPERLINK_IP` to the IP of the machine running `start_superlink.sh`.  
Leave it as `127.0.0.1` if everything runs on the same machine.

In [2]:
SUPERLINK_IP = "127.0.0.1"
SUPERLINK_FLEET_PORT = 9092
SUPERLINK_CONTROL_PORT = 9093

---
## 2 — Quick local simulation (no external machines needed)

Runs 2 virtual SuperNodes in-process. MNIST will be downloaded on first run (~11 MB).
Each epoch takes longer than the synthetic demo — expect ~1-2 min per round on CPU.

In [3]:
command = r"""
flwr run fl_app_mnist local \
--run-config num-server-rounds=5 \
--run-config min-clients=2 \
--run-config num-partitions=2 \
--run-config max-samples=500 \
--federation-config num-supernodes=2 \
--stream
""".strip()

rc = run_stream(flwr_bin_prefix + command, cwd=ctx.root)
if rc != 0:
    raise RuntimeError(f"Local MNIST simulation failed with exit code {rc}")

$ /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/.venv-workshop/bin/flwr run fl_app_mnist local \
  --run-config num-server-rounds=5 \
  --run-config min-clients=2 \
  --run-config num-partitions=2 \
  --run-config max-samples=500 \
  --federation-config num-supernodes=2 \
  --stream


🎊 Successfully started run 9481811888322884121


INFO :      Starting logstream for run_id `9481811888322884121`


INFO :      Starting Flower Simulation


INFO :      Starting Flower ServerApp, config: num_rounds=5, no round_timeout


...

INFO :      		round 3: 0.5206951547414065


INFO :      		round 4: 0.3844436412677169


INFO :      		round 5: 0.34614640604704616


INFO :      


INFO :      


---
## 3 — Distributed federation

### 3a — Verify connectivity

Make sure the SuperLink is reachable before launching the run.

In [4]:
superlink_ready = print_port_status(
    SUPERLINK_IP,
    {"Fleet API": SUPERLINK_FLEET_PORT, "Control API": SUPERLINK_CONTROL_PORT},
)

Fleet API    127.0.0.1:9092  -> not reachable
Control API  127.0.0.1:9093  -> not reachable


## Distributed Run: Deployment Runtime

This follows Flower's Deployment Runtime flow:

1. Start one `flower-superlink` process.
2. Start two `flower-supernode` processes and pass `--node-config` so each SuperNode gets a distinct partition.
3. Add/use a named SuperLink connection, `local-deployment`, pointing to the SuperLink Control API.
4. Submit the MNIST app with `flwr run ... local-deployment --stream`.

For a real deployment, run the SuperLink and SuperNodes on their own machines and use TLS instead of `--insecure`.

In [5]:
SUPERLINK_IP = "127.0.0.1"
SUPERLINK_FLEET_PORT = 9092
SUPERLINK_CONTROL_PORT = 9093
DEPLOYMENT_CONNECTION = "local-deployment"

superlink_ready = print_port_status(
    SUPERLINK_IP,
    {"Fleet API": SUPERLINK_FLEET_PORT, "Control API": SUPERLINK_CONTROL_PORT},
)

command = r"""
flwr run fl_app_mnist local-deployment \
--run-config num-server-rounds=5 \
--run-config min-clients=2 \
--run-config num-partitions=2 \
--run-config max-samples=500 \
--stream
""".strip()

if superlink_ready:
    update_superlink_address(f"{SUPERLINK_IP}:{SUPERLINK_CONTROL_PORT}", name=DEPLOYMENT_CONNECTION)
    rc = run_stream(flwr_bin_prefix + command, cwd=ctx.root)
    if rc != 0:
        raise RuntimeError(f"Distributed MNIST run failed with exit code {rc}")
else:
    print("SuperLink is not reachable; skipping distributed run.")
    print("Command to run after SuperLink and SuperNodes are started:")
    print(command)

Fleet API    127.0.0.1:9092  -> not reachable
Control API  127.0.0.1:9093  -> not reachable
SuperLink is not reachable; skipping distributed run.
Command to run after SuperLink and SuperNodes are started:
flwr run fl_app_mnist local-deployment \
--run-config num-server-rounds=5 \
--run-config min-clients=2 \
--run-config num-partitions=2 \
--run-config max-samples=500 \
--stream


---
## 4 — Optional: run everything locally for testing

If you want to test the distributed path **without multiple machines**,  
this cell starts a SuperLink and two SuperNodes as background subprocesses  
on this machine, waits for them to register, then runs the federation.

In [6]:
RUN_LOCAL_DEPLOYMENT_DEMO = False

superlink_command = r"""
flower-superlink \
--insecure
""".strip()

supernode_0_command = r"""
flower-supernode \
--insecure \\
--superlink 127.0.0.1:9092 \\
--clientappio-api-address 127.0.0.1:9094 \\
--node-config "partition-id=0 num-partitions=2"
""".strip()

supernode_1_command = r"""
flower-supernode \
--insecure \\
--superlink 127.0.0.1:9092 \\
--clientappio-api-address 127.0.0.1:9095 \\
--node-config "partition-id=1 num-partitions=2"
""".strip()

if RUN_LOCAL_DEPLOYMENT_DEMO:
    procs = []
    procs.append(start_background(flwr_bin_prefix + superlink_command, label="SuperLink", cwd=ctx.root))
    procs.append(start_background(flwr_bin_prefix + supernode_0_command, label="SuperNode-0", cwd=ctx.root))
    procs.append(start_background(flwr_bin_prefix + supernode_1_command, label="SuperNode-1", cwd=ctx.root))
    print("Run the distributed cell above, then stop the background processes with:")
    print("stop_background(procs)")
else:
    print("Set RUN_LOCAL_DEPLOYMENT_DEMO = True to start these background commands:")
    for command in [superlink_command, supernode_0_command, supernode_1_command]:
        print()
        print(command)

Set RUN_LOCAL_DEPLOYMENT_DEMO = True to start these background commands:

flower-superlink \
--insecure

flower-supernode \
--insecure \\
--superlink 127.0.0.1:9092 \\
--clientappio-api-address 127.0.0.1:9094 \\
--node-config "partition-id=0 num-partitions=2"

flower-supernode \
--insecure \\
--superlink 127.0.0.1:9092 \\
--clientappio-api-address 127.0.0.1:9095 \\
--node-config "partition-id=1 num-partitions=2"
